# EWS-CESM: análisis según protocolo congelado v1.0

Implementa `PROTOCOL_EWS_CESM_FROZEN.md` sobre el colapso cuasi-estático CESM
(van Westen, Kliphuis & Dijkstra 2024; Zenodo 10.5281/zenodo.10461549, CC-BY 4.0).

**Antes de ejecutar:** el protocolo debe tener timestamp público (commit/OSF/Zenodo).
Este notebook no cambia reglas: umbrales, regla de alarma, gates y sensibilidad están
fijados en el protocolo. Cualquier desviación se documenta como enmienda.

Requisitos: `numpy pandas scipy netCDF4 matplotlib` y los módulos locales
`cesm_ingest.py`, `ews_analysis.py`. Descarga total ≈ 460 MB (44 períodos).

In [ ]:
# 0-bis) Localización de módulos del paquete (ejecutar primero)
import sys, os
from pathlib import Path

# Si el notebook NO está dentro de la carpeta EWS_CESM_MZ_experiment/,
# indica aquí la ruta donde descomprimiste el zip:
PKG_DIR = ""   # p.ej. r"C:/Users/tu_usuario/EWS_CESM_MZ_experiment" o "/home/tu_usuario/EWS_CESM_MZ_experiment"

candidatos = [Path.cwd()] + ([Path(PKG_DIR)] if PKG_DIR else []) + [
    Path.cwd()/"EWS_CESM_MZ_experiment",
    Path.home()/"EWS_CESM_MZ_experiment",
    Path.home()/"Downloads"/"EWS_CESM_MZ_experiment",
]
pkg = next((p for p in candidatos if (p/"cesm_ingest.py").exists() and (p/"ews_analysis.py").exists()), None)
if pkg is None:
    raise ModuleNotFoundError(
        "No encuentro cesm_ingest.py / ews_analysis.py.\n"
        "Solución: descomprime EWS_CESM_MZ_experiment_v1_1.zip y (a) coloca este notebook "
        "DENTRO de esa carpeta, o (b) escribe la ruta de la carpeta en PKG_DIR arriba y reejecuta. "
        f"Rutas revisadas: {[str(c) for c in candidatos]}")
os.chdir(pkg)                       # cache_cesm/ y outputs_run/ quedan dentro del paquete
sys.path.insert(0, str(pkg))
print("Paquete encontrado y directorio de trabajo fijado en:", pkg)

In [ ]:
# 0) Configuración
from pathlib import Path
import numpy as np, pandas as pd, json
import matplotlib.pyplot as plt
import cesm_ingest as ci
import ews_analysis as ea

CACHE = Path("cache_cesm"); OUT = Path("outputs_run"); OUT.mkdir(exist_ok=True)
RUN_SENSITIVITY = False     # gate E4: activar tras la corrida primaria
np.random.seed(0)
print("periodos:", len(ci.PERIODS), "| tipping:", ea.TIPPING_YEAR,
      "| control:", ea.CONTROL_YEARS, "| eval:", ea.EVAL_YEARS)

In [ ]:
# 1) Ingesta completa (44 periodos, con cache local y hashes SHA-256)
csv = ci.build_contract_table(ci.PERIODS, CACHE, OUT, with_fov=True)
df = pd.read_csv(csv)
npz = np.load(OUT/"cesm_profiles_500_4500.npz")
profiles = npz["profile"]
print(df.shape, "| años:", df.year.min(), "-", df.year.max())
df.head(3)

In [ ]:
# 2) Gate E1 — paridad del loader contra las series procesadas del release
import urllib.request
from netCDF4 import Dataset
base = ("https://raw.githubusercontent.com/RenevanWesten/SA-AMOC-Collapse/"
        "SA-AMOC-Collapse_v1.0/Data/CESM/Ocean")
for name in ["AMOC_transport_depth_0-1000m.nc", "FOV_index_section_34S.nc"]:
    p = CACHE/("ref_"+name)
    if not p.exists():
        urllib.request.urlretrieve(f"{base}/{name}", p)
r = Dataset(CACHE/"ref_AMOC_transport_depth_0-1000m.nc"); a_ref = np.asarray(r["Transport"][:]); r.close()
r = Dataset(CACHE/"ref_FOV_index_section_34S.nc"); f_ref = np.asarray(r["F_OV"][:]); r.close()
dA = float(np.abs(df.amoc_transport_0_1000m_26N_Sv.values - a_ref).max())
dF = float(np.abs(df.FovS_Sv.values - f_ref).max())
E1 = {"max_abs_diff_AMOC_Sv": dA, "max_abs_diff_FovS_Sv": dF, "passed": bool(dA<=1e-10 and dF<=1e-10)}
print(E1); assert E1["passed"], "E1 FALLA: revisar loader antes de continuar" 

In [ ]:
# 3) Panorama: índice, F_H y F_ovS
fig, ax = plt.subplots(2,1, figsize=(10,6), sharex=True)
ax[0].plot(df.year, df.amoc_transport_0_1000m_26N_Sv, lw=.8)
ax[0].axvline(ea.TIPPING_YEAR, color='r', ls='--'); ax[0].axvspan(*ea.CONTROL_YEARS, alpha=.12, color='g')
ax[0].set_ylabel('AMOC 26N [Sv]')
ax[1].plot(df.year, df.FovS_Sv, lw=.8, color='tab:orange', label='FovS')
ax[1].plot(df.year, df.F_H_Sv, lw=.8, color='k', label='F_H')
ax[1].axvline(ea.TIPPING_YEAR, color='r', ls='--'); ax[1].legend(); ax[1].set_xlabel('año-modelo')
plt.tight_layout(); plt.savefig(OUT/'fig1_overview.png', dpi=150); plt.show()

In [ ]:
# 4) C1 — AR(1) + varianza (ventana causal 100 años, detrend lineal intra-ventana)
c1 = ea.ar1_variance(df.amoc_transport_0_1000m_26N_Sv.values, df.year.values)
c1.to_csv(OUT/'c1_ar1_variance.csv', index=False); c1.tail(3)

In [ ]:
# 5) C3 — diagnóstico MZ en el control (para calibrar umbral; causal, cada 5 años)
mz_ctrl = ea.mz_control_diagnostics(df, profiles)
mz_ctrl.to_csv(OUT/'c3_mz_control.csv', index=False); mz_ctrl.describe()

In [ ]:
# 6) C3 — lazo causal completo (refit cada 25 años; ~55 refits; puede tardar)
mz = ea.mz_causal_diagnostics(df, profiles)
mz.to_csv(OUT/'c3_mz_diagnostics.csv', index=False); mz.tail(3)

## ENMIENDA 1 (v1.1) — evaluación corregida
Reglas A1–A4 del protocolo (ver `PROTOCOL_EWS_CESM_FROZEN.md`, sección Enmienda 1):
E3 con semántica correcta; alarma MZ a nivel de refit (2 refits sostenidos, no-retorno 1);
FovS con detección causal de punto de giro con retractación; trazabilidad de
hiperparámetros y robustez del episodio supercrítico.

In [ ]:
# 9) Verificación interna: exacto vs 2º orden (informativa; la expansión es asintótica en delta)
err = np.abs(mz.mz_exact_re - mz.mz_rho2)
print(f"|exacto-rho2| mediana={err.median():.2e}  p95={err.quantile(.95):.2e} (válida solo donde |delta| es pequeño)")

In [ ]:
# 10) Evaluación enmendada (A1-A3): umbrales, alarmas, leads, E2 y E3
results = {}
# --- MZ: regla A2 a nivel de refit ---
ry, rd = ea.refit_level_series(mz.year.values, mz.mz_exact_re.values)
thr_mz, p_mz = ea.calibrate_threshold_refit(mz_ctrl.year.values, mz_ctrl.mz_exact_re.values, +1)
fpr_mz = ea.fpr_per_century_refit(mz_ctrl.year.values, mz_ctrl.mz_exact_re.values, thr_mz, +1)
lead_mz, ya_mz = ea.lead_time_refit(ry, rd, thr_mz, +1)
results['MZ_mult'] = dict(threshold=thr_mz, percentile=float(p_mz), fpr_control=fpr_mz,
                          alarm_year=ya_mz, lead_years=lead_mz,
                          refits_supercriticos=[int(y) for y,v in zip(ry,rd) if v>=thr_mz])
# --- AR(1) y varianza: regla anual original ---
c1_ctrl = c1[c1.year<=ea.CONTROL_YEARS[1]]
for nom, col in [('AR1','ar1'), ('varianza','variance')]:
    thr, p = ea.calibrate_threshold(c1_ctrl[col].values, +1, years=c1_ctrl.year.values)
    fpr = ea.fpr_per_century(c1_ctrl.year.values, c1_ctrl[col].values, thr, +1)
    lead, ya = ea.lead_time(c1.year.values, c1[col].values, thr, +1)
    results[nom] = dict(threshold=thr, percentile=float(p), fpr_control=fpr,
                        alarm_year=ya, lead_years=lead)
# --- FovS: regla A3 con retractación ---
cF = df[df.year<=ea.CONTROL_YEARS[1]].FovS_Sv
lead_f, ya_f, nret, ret = ea.fov_causal_min_alarm(df.year.values, df.FovS_Sv.values,
                                                  float(cF.min()), float(cF.std()))
results['FovS'] = dict(alarm_year=ya_f, lead_years=lead_f, alarmas_retractadas=nret)
for k,v in results.items(): print(k, v)
E2 = {k: (v.get('fpr_control',0.0)<=ea.FPR_TARGET_PER_CENTURY) for k,v in results.items()}
E3 = ea.evaluate_E3(results['MZ_mult']['lead_years'], results['AR1']['lead_years'])
print('\nE2:', E2, '| E3 (A1):', E3)

In [ ]:
# 12) Gate E4 — sensibilidad con reglas enmendadas (refit_10 es la variante decisiva)
def rebuild_state_df(cfg):
    st = ci.vector_state_26n(ci.PERIODS, CACHE, cfg)
    d2 = df.copy()
    for col, key in [('amoc_max_Sv','amoc_max'), ('depth_of_max_m','depth_of_max_m'),
                     ('deep_return_min_Sv','deep_return_min'), ('upper_mean_Sv','upper_mean'),
                     ('deep_mean_Sv','deep_mean')]:
        d2[col] = st[key]
    return d2, st['profile_500_4500']

def e3_variant(df_v, prof_v, ar_window=ea.AR_WINDOW, refit_every=ea.REFIT_EVERY, save=None):
    c1v = ea.ar1_variance(df_v.amoc_transport_0_1000m_26N_Sv.values, df_v.year.values, window=ar_window)
    c1c = c1v[c1v.year<=ea.CONTROL_YEARS[1]]
    mzc = ea.mz_control_diagnostics(df_v, prof_v)
    mzv = ea.mz_causal_diagnostics(df_v, prof_v, refit_every=refit_every, progress=False)
    if save: mzv.to_csv(OUT/save, index=False)
    thr,_ = ea.calibrate_threshold_refit(mzc.year.values, mzc.mz_exact_re.values, +1)
    ry, rd = ea.refit_level_series(mzv.year.values, mzv.mz_exact_re.values)
    lmz, ymz = ea.lead_time_refit(ry, rd, thr, +1)
    ta,_ = ea.calibrate_threshold(c1c.ar1.values, +1, years=c1c.year.values)
    lar, yar = ea.lead_time(c1v.year.values, c1v.ar1.values, ta, +1)
    return {'lead_MZ': lmz, 'alarma_MZ': ymz, 'lead_AR1': lar,
            'E3': ea.evaluate_E3(lmz, lar), 'umbral_MZ': thr}

if RUN_SENSITIVITY:
    E4 = {}
    E4['refit_10']       = e3_variant(df, profiles, refit_every=10, save='c3_mz_diagnostics_refit10.csv')
    E4['ar_window_70']   = e3_variant(df, profiles, ar_window=70)
    d2, p2 = rebuild_state_df(ci.StateConfig(target_lat=26.5));           E4['lat_26.5'] = e3_variant(d2, p2)
    d3, p3 = rebuild_state_df(ci.StateConfig(deep_band_m=(1500.,4000.))); E4['deep_1500_4000'] = e3_variant(d3, p3)
    print(json.dumps(E4, indent=1, default=str))
else:
    E4 = None; print('E4 pendiente (RUN_SENSITIVITY=False).')

In [ ]:
# 12) Gate E4 — sensibilidad con reglas enmendadas (refit_10 es la variante decisiva)
def e3_variant(df_v, prof_v, ar_window=ea.AR_WINDOW, refit_every=ea.REFIT_EVERY):
    c1v = ea.ar1_variance(df_v.amoc_transport_0_1000m_26N_Sv.values, df_v.year.values, window=ar_window)
    c1c = c1v[c1v.year<=ea.CONTROL_YEARS[1]]
    mzc = ea.mz_control_diagnostics(df_v, prof_v)
    mzv = ea.mz_causal_diagnostics(df_v, prof_v, refit_every=refit_every, progress=False)
    thr,_ = ea.calibrate_threshold_refit(mzc.year.values, mzc.mz_exact_re.values, +1)
    ry, rd = ea.refit_level_series(mzv.year.values, mzv.mz_exact_re.values)
    lmz,_ = ea.lead_time_refit(ry, rd, thr, +1)
    ta,_ = ea.calibrate_threshold(c1c.ar1.values, +1, years=c1c.year.values)
    lar,_ = ea.lead_time(c1v.year.values, c1v.ar1.values, ta, +1)
    return {'lead_MZ':lmz, 'lead_AR1':lar, 'E3':ea.evaluate_E3(lmz,lar), 'mzv':mzv}
if RUN_SENSITIVITY:
    E4 = {}
    v = e3_variant(df, profiles, refit_every=10); E4['refit_10'] = {k:v[k] for k in ['lead_MZ','lead_AR1','E3']}
    v['mzv'].to_csv(OUT/'c3_mz_diagnostics_refit10.csv', index=False)
    E4['ar_window_70'] = {k:w[k] for k in ['lead_MZ','lead_AR1','E3'] if (w:=e3_variant(df, profiles, ar_window=70))}
    d2, p2 = rebuild_state_df(ci.StateConfig(target_lat=26.5))
    E4['lat_26.5'] = {k:w[k] for k in ['lead_MZ','lead_AR1','E3'] if (w:=e3_variant(d2, p2))}
    d3, p3 = rebuild_state_df(ci.StateConfig(deep_band_m=(1500.,4000.)))
    E4['deep_1500_4000'] = {k:w[k] for k in ['lead_MZ','lead_AR1','E3'] if (w:=e3_variant(d3, p3))}
    print(json.dumps(E4, indent=1, default=str))
else:
    E4 = None; print('E4 pendiente (RUN_SENSITIVITY=False).')

def rebuild_state_df(cfg):
    st = ci.vector_state_26n(ci.PERIODS, CACHE, cfg)
    d2 = df.copy()
    for col, key in [('amoc_max_Sv','amoc_max'), ('depth_of_max_m','depth_of_max_m'),
                     ('deep_return_min_Sv','deep_return_min'), ('upper_mean_Sv','upper_mean'),
                     ('deep_mean_Sv','deep_mean')]:
        d2[col] = st[key]
    return d2, st['profile_500_4500']

In [ ]:
# 13) Resultados v1.1 -> JSON
summary = {'protocol':'PROTOCOL_EWS_CESM_FROZEN.md v1.0 + Enmienda 1 (v1.1)',
           'E1_parity': E1, 'E2_fpr': {k:bool(v) for k,v in E2.items()},
           'E3': E3, 'per_method': results, 'E4': E4,
           'FovS_reference_leads': {'spline_min_1732': 26.0, 'raw_min_1726': 32.0}}
json.dump(summary, open(OUT/'RESULTS_SUMMARY_v1_1.json','w'), indent=2, default=str)
print(json.dumps(summary, indent=2, default=str))